In [61]:
import numpy as np
import pandas as pd
from datetime import datetime
from datetime import timedelta
import math

In [2]:
df = pd.read_parquet("../data/raw/EURUSD_H1.parquet")

In [4]:
df.index.tz

zoneinfo.ZoneInfo(key='UTC')

In [5]:
ny_hour = df.index.tz_convert("America/New_York")

In [10]:
df

,open,high,low,close,volume
timestamp,,,,,
2025-01-01 22:00:00+00:00,1.03503,1.03577,1.03483,1.03550,1443.22
2025-01-01 23:00:00+00:00,1.03550,1.03563,1.03462,1.03507,4447.37
2025-01-02 00:00:00+00:00,1.03508,1.03564,1.03472,1.03520,3681.37
2025-01-02 01:00:00+00:00,1.03521,1.03606,1.03442,1.03578,5012.45
2025-01-02 02:00:00+00:00,1.03577,1.03712,1.03577,1.03709,2791.47
...,...,...,...,...,...
2025-12-31 17:00:00+00:00,1.17346,1.17390,1.17312,1.17358,3662.76
2025-12-31 18:00:00+00:00,1.17357,1.17386,1.17302,1.17357,3944.61
2025-12-31 19:00:00+00:00,1.17356,1.17432,1.17356,1.17407,3321.68


In [ ]:
h4 = df.resample(
    "4h",
    origin="start",
    closed="left",
    label="left"
).agg({
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum"
})

In [31]:
d1 = df.resample(
    "D",
    origin="start",
    closed="left",
    label="left"
).agg({
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum"
})

C:\Users\dungt\AppData\Local\Temp\ipykernel_27244\85321318.py:1: RuntimeWarning: The 'origin' keyword does not take effect when resampling with a Day 'freq' unless the index is a timezone-naive DatetimeIndex
  d1 = df.resample(


In [30]:
print(f"Number 4hour candle and daily canlde: {len(h4)} and {len(d1)}")

Number 4hour candle and daily canlde: 2184 and 364


In [ ]:
ohlc = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}

trade_time = df.index.tz_convert("America/New_York") + pd.Timedelta(hours=7)
df_td = df.set_axis(trade_time)    # bảng mới với nhãn mới; df gốc vẫn UTC

size_b = df_td.resample("D").size()
d1 = df_td.resample("D").agg(ohlc)
d1 = d1[size_b > 0]                # bỏ ô thứ Bảy/Chủ nhật rỗng (NaN)

print(len(d1))
print(size_b[size_b > 0].value_counts().sort_index())
print(d1.index.day_name().value_counts())         

364
10      1
23      1
24    258
Name: count, dtype: int64
timestamp
Thursday     52
Friday       52
Saturday     52
Sunday       52
Monday       52
Tuesday      52
Wednesday    52
Name: count, dtype: int64


In [37]:
size_b

timestamp
2025-01-02 00:00:00-05:00    24
2025-01-03 00:00:00-05:00    24
2025-01-04 00:00:00-05:00     0
2025-01-05 00:00:00-05:00     0
2025-01-06 00:00:00-05:00    24
                             ..
2025-12-27 00:00:00-05:00     0
2025-12-28 00:00:00-05:00     0
2025-12-29 00:00:00-05:00    24
2025-12-30 00:00:00-05:00    24
2025-12-31 00:00:00-05:00    24
Freq: D, Length: 364, dtype: int64

In [56]:
ma20 = df["close"].rolling(20).mean()
std20 = df["close"].rolling(20).std()

In [57]:
print(ma20.isna().sum()) # 19

19


In [ ]:
math.isclose(ma20.iloc[19],df["close"].iloc[0:20].mean()) # Có bằng nhau

True

In [66]:
np.std(df["close"].iloc[0:20])

np.float64(0.003525474826175899)

In [ ]:
std20.iloc[19] # Không bằng nhau

np.float64(0.003617060852477693)

In [ ]:
def returns(df):
    return df / df.shift(1)

In [72]:
df["ret"] = df["close"].pct_change()

In [78]:
signal = pd.Series(np.where(df["close"]> ma20,1,-1),index = df.index)
signal

timestamp
2025-01-01 22:00:00+00:00   -1
2025-01-01 23:00:00+00:00   -1
2025-01-02 00:00:00+00:00   -1
2025-01-02 01:00:00+00:00   -1
2025-01-02 02:00:00+00:00   -1
                            ..
2025-12-31 17:00:00+00:00   -1
2025-12-31 18:00:00+00:00   -1
2025-12-31 19:00:00+00:00    1
2025-12-31 20:00:00+00:00    1
2025-12-31 21:00:00+00:00    1
Length: 6225, dtype: int64

In [ ]:
strat_ret_cheat = signal * df["ret"]
pnl = np.prod(1 + strat_ret_cheat)
print(pnl)

4.856850101471573


In [85]:
strat_ret = signal.shift(1) * df["ret"]
pnl = np.prod(1 + strat_ret)
print(pnl)

0.982273256104046
